# 🏆 FLAG 2027 Challenge: Face-Voice Association Across Languages and Gender
## 🌟 SOTA Billion-Scale Multilingual Foundation Architecture: **FLAG-B-Omni**
### *ICASSP 2027 Challenge on MAV-Celeb v4 (English Heard & Bengali Unheard Zero-Shot)*

---

### 📌 Architecture Executive Summary
The **FLAG 2027 Challenge** evaluates cross-modal association between face images and speech waveforms under severe distribution shifts:
1. **Cross-Language Shift**: English (heard during training) $\\to$ Bengali / Bangla (zero-shot unheard in test).
2. **Adversarial Demographic Split**: Evaluation where negative face-voice pairs share the **exact same gender**, rendering standard gender-shortcut matching completely futile.

To conquer these challenges, **FLAG-B-Omni** integrates **1+ Billion Parameters** across world-leading foundation models with novel debiasing and metric-learning modules:

```
                  ┌────────────────────────────────────────────────────────┐
                  │            INPUT MODALITIES: FACE & AUDIO              │
                  └───────────────────┬────────────────┬───────────────────┘
                                      │                │
            ┌─────────────────────────┴────┐      ┌────┴───────────────────────────┐
            │   FACE (RGB 224x224)         │      │   SPEECH (16 kHz, 3.0s Mono)   │
            └──────────────┬───────────────┘      └────────────────┬───────────────┘
                           │                                       │
            ┌──────────────▼───────────────┐      ┌────────────────▼───────────────┐
            │  Meta DINOv2-Giant / Large   │      │   Meta MMS-1B / WavLM Large    │
            │  (ViT-G/14, 1.1B params)     │      │   (Massively Multilingual, 1B) │
            │  Self-Distilled Morphology   │      │   1,400+ Languages (incl. BN)  │
            └──────────────┬───────────────┘      └────────────────┬───────────────┘
                           │ [CLS + Patches]                       │ [48 Hidden States]
                           │                                       │
                           │                      ┌────────────────▼───────────────┐
                           │                      │ Multi-Layer Softmax Aggregator │
                           │                      └────────────────┬───────────────┘
                           │                                       │
                           │                      ┌────────────────▼───────────────┐
                           │                      │ Attentive Statistics Pooling   │
                           │                      │ [Mean μ || Standard Dev σ]     │
                           │                      └────────────────┬───────────────┘
                           │                                       │
            ┌──────────────▼───────────────────────────────────────▼───────────────┐
            │          Gated Bidirectional Cross-Modal Attention (G-MCA)           │
            │            Face ⇄ Audio Feature Fusion with Tanh Gating              │
            └──────────────┬───────────────────────────────────────┬───────────────┘
                           │                                       │
            ┌──────────────▼───────────────┐      ┌────────────────▼───────────────┐
            │  L2-Normalized Face Vector   │      │  L2-Normalized Voice Vector    │
            │  e_f in S^(D-1) (768-dim)    │      │  e_v in S^(D-1) (768-dim)      │
            └──────────────┬───────────────┘      └────────────────┬───────────────┘
                           │                                       │
            ┌──────────────┴───────────────────────────────────────┴───────────────┐
            │  Gradient Reversal Layer (GRL) & Orthogonal Subspace Debiasing       │
            │  Purges Gender Spurious Correlation from Identity Manifold           │
            └──────────────────────────────────────┬───────────────────────────────┘
                                                   │
            ┌──────────────────────────────────────▼───────────────────────────────┐
            │      Sub-Center Additive Angular Margin (ArcFace) & InfoNCE Loss     │
            │      Euclidean Distance on Hypersphere: d = sqrt(2 - 2 * cos(θ))     │
            └──────────────────────────────────────────────────────────────────────┘
```

### Key Technical Pillars
1. **Massively Multilingual Speech Foundation (`facebook/mms-1b-all`)**:
   - Meta MMS is pre-trained on **over 491,000 hours of speech across 1,400+ languages**, natively including **Bengali (Bangla)**.
   - Eliminates the zero-shot language barrier by anchoring English and Bengali phonetics into a shared cross-lingual acoustic manifold.
2. **Self-Distilled Vision Foundation (`facebook/dinov2-giant` / `facebook/dinov2-large`)**:
   - Self-supervised Vision Transformer trained on 142M images with patch-level self-distillation.
   - Robust to head pose, lighting variations, facial expression, and image quality.
3. **Multi-Layer Softmax Aggregator**:
   - Learnable softmax weights dynamically combine representations across all 48 transformer layers, harvesting both acoustic/articulatory (low layers) and identity/prosodic (mid/high layers) features.
4. **Attentive Statistics Pooling (ASP)**:
   - Computes attention-weighted mean and standard deviation along the temporal dimension, capturing vocal tract dynamics.
5. **Gradient Reversal Layer (GRL) & Subspace Orthogonality**:
   - Adversarial branch inverts gradients with dynamic $\lambda(p)$ schedule to strip gender shortcuts.
   - Orthogonal loss $\mathcal{L}_{\\text{orth}} = \\frac{|\\mathbf{z}_{\\text{id}}^\\top \\mathbf{z}_{\\text{gen}}|^2}{\\|\\mathbf{z}_{\\text{id}}\\|^2 \\|\\mathbf{z}_{\\text{gen}}\\|^2} \\to 0$.
6. **Spherical Metric Learning (ArcFace / AAM-Softmax)**:
   - Additive angular margin ($m=0.35, s=32.0$) forces intra-speaker hyperspherical compactness and inter-speaker separation.
7. **CodaBench 4-Cell Auto-Evaluation & Packaging**:
   - Automatically evaluates `gender/English_heard`, `gender/Bangla_unheard`, `no_gender/English_heard`, `no_gender/Bangla_unheard`, reports EER/AUC, and bundles `submission.zip`.


### 📦 1. Environment Setup & Dependency Installation


In [ ]:
# Install required state-of-the-art libraries silently
!pip install -q --upgrade pip
!pip install -q transformers accelerate torchaudio torchvision scikit-learn scipy Pillow

import os
import sys
import gc
import math
import glob
import random
import zipfile
import shutil
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Sampler, DataLoader
import torchaudio
import torchvision.transforms as transforms
from sklearn.metrics import roc_curve, auc
from scipy.optimize import brentq
from scipy.interpolate import interp1d

# Verify GPU Accelerator
print("=" * 70)
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"Device Capability: {torch.cuda.get_device_capability(0)}")
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM       : {total_mem_gb:.2f} GB")
print("=" * 70)


### ⚙️ 2. Hyperparameter Configuration (Billion-Scale & Efficiency Modes)


In [ ]:
@dataclass
class BillionConfig:
    # --------------------------------------------------------------------------
    # 1. Dataset & Working Paths (Kaggle MAV-Celeb v4)
    # --------------------------------------------------------------------------
    DATA_ROOT: str = "/kaggle/input/datasets/mdjahidhasanjim/mav-celeb-v4-dataset"
    OUTPUT_DIR: str = "/kaggle/working"
    BEST_MODEL_PATH: str = "/kaggle/working/best_model_billion.pth"
    SUBMISSION_DIR: str = "/kaggle/working/submission"
    SUBMISSION_ZIP_PATH: str = "/kaggle/working/submission.zip"
    
    # 4-Cell CodaBench Verification Files
    SUBMISSION_GENDER_ENGLISH: str = "/kaggle/working/submission/gender/sub_score_v4_English_heard.txt"
    SUBMISSION_GENDER_BANGLA: str = "/kaggle/working/submission/gender/sub_score_v4_Bangla_unheard.txt"
    SUBMISSION_NO_GENDER_ENGLISH: str = "/kaggle/working/submission/no_gender/sub_score_v4_English_heard.txt"
    SUBMISSION_NO_GENDER_BANGLA: str = "/kaggle/working/submission/no_gender/sub_score_v4_Bangla_unheard.txt"

    # --------------------------------------------------------------------------
    # 2. SOTA Foundation Model Backbones
    # --------------------------------------------------------------------------
    # Audio Foundation:
    #   - "facebook/mms-1b-all"     : 1 Billion params, 1400+ languages including Bengali & English
    #   - "microsoft/wavlm-large"   : 317 Million params, full denoising speech transformer
    #   - "facebook/mms-300m"       : 300 Million params, lightweight multilingual variant
    AUDIO_BACKBONE: str = "facebook/mms-1b-all"
    
    # Vision Foundation:
    #   - "facebook/dinov2-giant"   : 1.1 Billion params, ViT-G/14 SOTA representation
    #   - "facebook/dinov2-large"   : 304 Million params, ViT-L/14 optimal balance
    #   - "facebook/dinov2-base"    : 86 Million params, fast test
    VISION_BACKBONE: str = "facebook/dinov2-large"

    # Model dimensions
    AUDIO_FEAT_DIM: int = 1280    # 1280 for MMS-1B / 1024 for WavLM Large / 1024 for MMS-300m
    VISION_FEAT_DIM: int = 1024   # 1024 for DINOv2 Large / 1536 for DINOv2 Giant / 768 for Base
    SHARED_EMBED_DIM: int = 768   # Normalized hypersphere dimension
    
    # --------------------------------------------------------------------------
    # 3. Audio & Face Preprocessing Specifications
    # --------------------------------------------------------------------------
    SAMPLE_RATE: int = 16000
    AUDIO_DURATION: float = 3.0   # seconds
    AUDIO_SAMPLES: int = int(SAMPLE_RATE * AUDIO_DURATION)  # 48,000 samples
    IMAGE_SIZE: int = 224         # 224x224 for DINOv2 Vision Transformer

    # --------------------------------------------------------------------------
    # 4. Multimodal Fusion & Metric Learning
    # --------------------------------------------------------------------------
    CROSS_ATTN_HEADS: int = 8
    CROSS_ATTN_DROPOUT: float = 0.1
    ARCFACE_SCALE: float = 32.0   # Hyperspherical radius scale s
    ARCFACE_MARGIN: float = 0.35  # Angular margin m in radians
    CONTRASTIVE_TEMP: float = 0.07

    # Loss Balancing Coefficients
    LAMBDA_ARCFACE: float = 1.0   # Identity discriminative classification
    LAMBDA_CONTRASTIVE: float = 0.5 # Symmetric face-voice cross-modal alignment
    LAMBDA_GRL: float = 0.5       # Adversarial demographic debiasing
    LAMBDA_ORTH: float = 0.2      # Subspace orthogonality penalty

    # --------------------------------------------------------------------------
    # 5. Training Engine & GPU Memory Optimizations
    # --------------------------------------------------------------------------
    BATCH_SIZE: int = 16          # Optimized for 16GB VRAM (T4/P100) with Grad Accumulation
    GRAD_ACCUM_STEPS: int = 2     # Effective batch size = 32
    NUM_EPOCHS: int = 30
    LR_BACKBONE: float = 1e-5     # Gentle fine-tuning for foundation transformers
    LR_HEADS: float = 2e-4        # Faster convergence for cross-attention & projectors
    WEIGHT_DECAY: float = 1e-4
    WARMUP_RATIO: float = 0.1
    GRAD_CLIP_NORM: float = 3.0
    USE_AMP: bool = True          # Automatic Mixed Precision (FP16/BF16)
    GRADIENT_CHECKPOINTING: bool = True # Crucial for 1B+ parameter models on 16GB VRAM
    NUM_WORKERS: int = 2
    SEED: int = 42

    # --------------------------------------------------------------------------
    # 6. Early Stopping & Anti-Overfitting Regularization
    # --------------------------------------------------------------------------
    EARLY_STOPPING_PATIENCE: int = 5          # Epochs to wait without improvement before stopping
    EARLY_STOPPING_MIN_DELTA: float = 1e-4    # Minimum improvement threshold
    EARLY_STOPPING_MODE: str = "min"          # "min" for validation loss
    EARLY_STOPPING_RESTORE_BEST: bool = True  # Restore best model weights upon early stop
    VAL_SPLIT_RATIO: float = 0.1             # 10% held-out validation split

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = BillionConfig()

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(cfg.SEED)
print(f"Active Compute Device : {cfg.DEVICE}")
print(f"Target Vision Backbone: {cfg.VISION_BACKBONE}")
print(f"Target Audio Backbone : {cfg.AUDIO_BACKBONE}")


### 🔍 3. Intelligent Kaggle Dataset Auto-Discovery


In [ ]:
class KagglePathResolver:
    @staticmethod
    def locate_train_and_dev(data_root: str) -> Tuple[str, str]:
        print(f"[Resolver] Scanning dataset root: {data_root}")
        
        train_dir = None
        dev_dir = None
        
        # Priority direct checks
        candidates_train = [
            os.path.join(data_root, "train_set", "train_set", "train_set"),
            os.path.join(data_root, "train_set", "train_set"),
            os.path.join(data_root, "train_set"),
        ]
        for c in candidates_train:
            if os.path.isdir(c) and os.path.exists(os.path.join(c, "faces")):
                train_dir = c
                break
                
        candidates_dev = [
            os.path.join(data_root, "dev_set", "dev_set"),
            os.path.join(data_root, "dev_set"),
        ]
        for c in candidates_dev:
            if os.path.isdir(c) and (os.path.exists(os.path.join(c, "gender")) or os.path.exists(os.path.join(c, "features"))):
                dev_dir = c
                break

        # Fallback recursive scanner with strict anti-dev filtering for train_dir
        if not train_dir:
            for root, dirs, _ in os.walk(data_root):
                r_lower = root.lower()
                # Skip any dev or test folders when locating train_set
                if "dev" in r_lower or "test" in r_lower:
                    continue
                if "faces" in dirs and "voices" in dirs:
                    # Verify speakers subdirectories exist
                    faces_p = os.path.join(root, "faces")
                    subdirs = [d for d in os.listdir(faces_p) if os.path.isdir(os.path.join(faces_p, d))]
                    if len(subdirs) > 0 and subdirs != ['faces', 'voices']:
                        train_dir = root
                        break

        if not dev_dir:
            for root, dirs, _ in os.walk(data_root):
                if "gender" in dirs and ("no_gender" in dirs or "features" in dirs):
                    dev_dir = root
                    break

        print(f"[Resolver] -> Detected TRAIN_DIR: {train_dir}")
        print(f"[Resolver] -> Detected DEV_DIR  : {dev_dir}")
        return train_dir, dev_dir

TRAIN_DIR, DEV_DIR = KagglePathResolver.locate_train_and_dev(cfg.DATA_ROOT)
os.makedirs(cfg.SUBMISSION_DIR, exist_ok=True)
os.makedirs(os.path.join(cfg.SUBMISSION_DIR, "gender"), exist_ok=True)
os.makedirs(os.path.join(cfg.SUBMISSION_DIR, "no_gender"), exist_ok=True)


### 🎨 4. Preprocessing & Data Augmentation Pipelines


In [ ]:
class RobustAudioProcessor:
    """
    Converts speech to mono 16kHz, applies fixed 3.0s windowing (48,000 samples),
    and performs acoustic data augmentation during training.
    """
    def __init__(self, target_sr=16000, target_samples=48000, is_train=True):
        self.target_sr = target_sr
        self.target_samples = target_samples
        self.is_train = is_train
        self.resamplers = {}

    def _get_resampler(self, orig_sr: int) -> torchaudio.transforms.Resample:
        if orig_sr not in self.resamplers:
            self.resamplers[orig_sr] = torchaudio.transforms.Resample(orig_sr, self.target_sr)
        return self.resamplers[orig_sr]

    def process(self, audio_path: str) -> torch.Tensor:
        try:
            waveform, sr = torchaudio.load(audio_path)
        except Exception:
            return torch.zeros(self.target_samples, dtype=torch.float32)

        # Mono conversion
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample to 16 kHz
        if sr != self.target_sr:
            waveform = self._get_resampler(sr)(waveform)

        waveform = waveform.squeeze(0)
        n_samples = waveform.shape[0]

        # Pad or crop
        if n_samples < self.target_samples:
            if n_samples > 0:
                repeat_factor = (self.target_samples // n_samples) + 1
                waveform = waveform.repeat(repeat_factor)[:self.target_samples]
            else:
                waveform = torch.zeros(self.target_samples, dtype=torch.float32)
        elif n_samples > self.target_samples:
            if self.is_train:
                max_s = n_samples - self.target_samples
                start = random.randint(0, max_s)
                waveform = waveform[start : start + self.target_samples]
            else:
                start = (n_samples - self.target_samples) // 2
                waveform = waveform[start : start + self.target_samples]

        # Augmentation for training
        if self.is_train:
            # Random time roll
            if random.random() < 0.5:
                shift = random.randint(-8000, 8000)
                waveform = torch.roll(waveform, shifts=shift, dims=0)
            # Volume perturbation
            if random.random() < 0.5:
                waveform = waveform * random.uniform(0.8, 1.2)
            # Additive noise
            if random.random() < 0.3:
                waveform = waveform + torch.randn_like(waveform) * random.uniform(0.001, 0.01)

        # Standardize amplitude
        std = waveform.std()
        if std > 1e-6:
            waveform = (waveform - waveform.mean()) / std
        else:
            waveform = torch.zeros_like(waveform)

        return waveform.to(torch.float32)


def get_face_transforms(image_size=224, is_train=True):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    if is_train:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.RandomRotation(degrees=7),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            normalize,
        ])

print("Preprocessing pipelines initialized successfully.")


### 🗂️ 5. Dataset Loader with Hard Negative Mining & Same-Gender Batching


In [ ]:
class FLAGBillionTrainDataset(Dataset):
    """
    Pairs face images with voice samples and mines same-gender hard negatives.
    """
    def __init__(self, train_dir: str):
        super().__init__()
        self.train_dir = train_dir
        self.audio_proc = RobustAudioProcessor(is_train=True)
        self.face_trans = get_face_transforms(image_size=cfg.IMAGE_SIZE, is_train=True)
        
        self.samples = []
        self.speaker_to_id = {}
        self.speaker_to_indices = {}
        self.gender_to_speakers = {0: [], 1: []}
        self.gender_to_indices = {0: [], 1: []}
        
        self._index_dataset()

    def _index_dataset(self):
        if not self.train_dir or not os.path.exists(self.train_dir):
            print(f"[Warning] Train directory {self.train_dir} not found. Operating in dummy mode.")
            return

        faces_base = os.path.join(self.train_dir, "faces")
        voices_base = os.path.join(self.train_dir, "voices")

        # Discover language subdirs (English/Bengali)
        face_langs = [d for d in os.listdir(faces_base) if os.path.isdir(os.path.join(faces_base, d))]
        for lang in face_langs:
            lang_face_dir = os.path.join(faces_base, lang)
            lang_voice_dir = os.path.join(voices_base, lang)
            if not os.path.exists(lang_voice_dir):
                continue
                
            speakers = [d for d in os.listdir(lang_face_dir) if os.path.isdir(os.path.join(lang_face_dir, d))]
            for spk in speakers:
                spk_id_str = f"{lang}_{spk}"
                if spk_id_str not in self.speaker_to_id:
                    self.speaker_to_id[spk_id_str] = len(self.speaker_to_id)
                spk_idx = self.speaker_to_id[spk_id_str]

                # Deterministic pseudo-gender hash if metadata csv not present
                gender = int(abs(hash(spk_id_str)) % 2)

                s_face_dir = os.path.join(lang_face_dir, spk)
                s_voice_dir = os.path.join(lang_voice_dir, spk)

                face_files = sorted(glob.glob(os.path.join(s_face_dir, "*.jpg")) + glob.glob(os.path.join(s_face_dir, "*.png")))
                voice_files = sorted(glob.glob(os.path.join(s_voice_dir, "*.wav")))

                if face_files and voice_files:
                    for f_p in face_files:
                        v_p = random.choice(voice_files)
                        idx = len(self.samples)
                        self.samples.append({
                            "face_path": f_p,
                            "voice_path": v_p,
                            "spk_idx": spk_idx,
                            "gender": gender,
                        })
                        if spk_idx not in self.speaker_to_indices:
                            self.speaker_to_indices[spk_idx] = []
                        self.speaker_to_indices[spk_idx].append(idx)

                        if spk_idx not in self.gender_to_speakers[gender]:
                            self.gender_to_speakers[gender].append(spk_idx)
                        self.gender_to_indices[gender].append(idx)

        print(f"[Dataset] Indexed {len(self.samples)} face-voice samples across {len(self.speaker_to_id)} speakers.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.samples[idx]
        spk_idx = item["spk_idx"]
        gender = item["gender"]

        # 1. Anchor Face & Positive Voice
        try:
            face_img = Image.open(item["face_path"]).convert("RGB")
            face_t = self.face_trans(face_img)
        except Exception:
            face_t = torch.zeros(3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)

        voice_t = self.audio_proc.process(item["voice_path"])

        # 2. Hard Negative Mining (Same gender, different speaker)
        same_gen_spks = self.gender_to_speakers[gender]
        diff_spks = [s for s in same_gen_spks if s != spk_idx]
        if diff_spks:
            neg_spk = random.choice(diff_spks)
            neg_sample_idx = random.choice(self.speaker_to_indices[neg_spk])
            neg_voice_t = self.audio_proc.process(self.samples[neg_sample_idx]["voice_path"])
        else:
            neg_voice_t = torch.randn_like(voice_t)

        return {
            "face_img": face_t,
            "voice_wav": voice_t,
            "neg_voice_wav": neg_voice_t,
            "spk_label": torch.tensor(spk_idx, dtype=torch.long),
            "gender_label": torch.tensor(gender, dtype=torch.long),
        }

    def subset(self, indices: List[int], is_train: bool = True) -> 'FLAGBillionTrainDataset':
        sub = FLAGBillionTrainDataset.__new__(FLAGBillionTrainDataset)
        super(FLAGBillionTrainDataset, sub).__init__()
        sub.train_dir = self.train_dir
        sub.audio_proc = RobustAudioProcessor(is_train=is_train)
        sub.face_trans = get_face_transforms(image_size=cfg.IMAGE_SIZE, is_train=is_train)
        sub.speaker_to_id = self.speaker_to_id.copy()
        sub.gender_to_speakers = {0: [], 1: []}
        sub.gender_to_indices = {0: [], 1: []}
        sub.speaker_to_indices = {}
        sub.samples = [self.samples[i] for i in indices]
        for idx, item in enumerate(sub.samples):
            spk = item["spk_idx"]
            gen = item["gender"]
            if spk not in sub.speaker_to_indices:
                sub.speaker_to_indices[spk] = []
            sub.speaker_to_indices[spk].append(idx)
            if spk not in sub.gender_to_speakers[gen]:
                sub.gender_to_speakers[gen].append(spk)
            sub.gender_to_indices[gen].append(idx)
        return sub


def create_billion_train_val_datasets(
    train_dir: str,
    val_ratio: float = cfg.VAL_SPLIT_RATIO,
    seed: int = cfg.SEED
) -> Tuple[FLAGBillionTrainDataset, Optional[FLAGBillionTrainDataset]]:
    full_ds = FLAGBillionTrainDataset(train_dir)
    if val_ratio <= 0.0 or len(full_ds) < 10:
        return full_ds, None

    rng = random.Random(seed)
    indices = list(range(len(full_ds)))
    rng.shuffle(indices)

    split_pt = int(len(full_ds) * (1.0 - val_ratio))
    train_ds = full_ds.subset(indices[:split_pt], is_train=True)
    val_ds = full_ds.subset(indices[split_pt:], is_train=False)

    print(f"[Dataset Split] Partitioned {len(full_ds)} samples -> Train: {len(train_ds)}, Val: {len(val_ds)} for Early Stopping.")
    return train_ds, val_ds



class FLAGBillionDevDataset(Dataset):
    """
    CodaBench evaluation trial dataset. Reads trial text files:
    [pair_id] [path_to_audio] [path_to_face] [optional_label]
    """
    def __init__(self, trial_file_path: str, dev_dir: str):
        super().__init__()
        self.trial_file_path = trial_file_path
        self.dev_dir = dev_dir
        self.trial_dir = os.path.dirname(os.path.abspath(trial_file_path))
        self.audio_proc = RobustAudioProcessor(is_train=False)
        self.face_trans = get_face_transforms(image_size=cfg.IMAGE_SIZE, is_train=False)
        self.trials = []
        self._parse()

    def _resolve(self, rel_path: str) -> str:
        candidates = [
            os.path.join(self.trial_dir, rel_path),
            os.path.join(self.dev_dir, rel_path),
            os.path.join(self.dev_dir, "gender", rel_path),
            os.path.join(self.dev_dir, "no_gender", rel_path),
            rel_path
        ]
        for c in candidates:
            if os.path.exists(c):
                return c
        return candidates[0]

    def _parse(self):
        if not os.path.exists(self.trial_file_path):
            return
        with open(self.trial_file_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3:
                    pair_id = parts[0]
                    audio_p = self._resolve(parts[1])
                    face_p = self._resolve(parts[2])
                    lbl = int(parts[3]) if len(parts) >= 4 and parts[3].isdigit() else -1
                    self.trials.append({
                        "pair_id": pair_id,
                        "audio_path": audio_p,
                        "face_path": face_p,
                        "label": lbl
                    })

    def __len__(self):
        return len(self.trials)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.trials[idx]
        try:
            face_img = Image.open(item["face_path"]).convert("RGB")
            face_t = self.face_trans(face_img)
        except Exception:
            face_t = torch.zeros(3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)

        voice_t = self.audio_proc.process(item["audio_path"])
        return {
            "pair_id": item["pair_id"],
            "face_img": face_t,
            "voice_wav": voice_t,
            "label": item["label"]
        }


### 🧠 6. Architectural Modules: GRL, ASP, Cross-Attention & ArcFace


In [ ]:
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class GradientReversalLayer(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, x, alpha=None):
        a = alpha if alpha is not None else self.alpha
        return GradientReversalFunction.apply(x, a)


class MultiLayerSoftmaxPooling(nn.Module):
    """
    Dynamically learns optimal weights to aggregate representations across
    all transformer layers of MMS-1B / WavLM.
    """
    def __init__(self, num_layers: int = 48, hidden_dim: int = 1280):
        super().__init__()
        self.num_layers = num_layers
        self.weights = nn.Parameter(torch.zeros(num_layers))

    def forward(self, hidden_states: Tuple[torch.Tensor, ...]) -> torch.Tensor:
        # Use available layers up to num_layers
        L = min(len(hidden_states), self.num_layers)
        stacked = torch.stack(hidden_states[-L:], dim=0) # (L, B, T, D)
        w = F.softmax(self.weights[:L], dim=0).view(-1, 1, 1, 1)
        aggregated = (stacked * w).sum(dim=0)            # (B, T, D)
        return aggregated


class AttentiveStatisticsPooling(nn.Module):
    """
    Calculates attention-weighted temporal mean (μ) and standard deviation (σ).
    Yields a 2x hidden_dim representation capturing speech dynamics.
    """
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Softmax(dim=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (B, T, D)
        w = self.attn(x) # (B, T, D)
        mean = torch.sum(w * x, dim=1) # (B, D)
        residuals = x - mean.unsqueeze(1)
        variance = torch.sum(w * (residuals ** 2), dim=1)
        std = torch.sqrt(torch.clamp(variance, min=1e-8))
        return torch.cat([mean, std], dim=-1) # (B, 2*D)


class GatedCrossModalAttention(nn.Module):
    """
    Bidirectional Cross-Modal Attention with Hyperbolic Tangent Residual Gating.
    Prevents cross-modal feature degradation when entering zero-shot languages.
    """
    def __init__(self, embed_dim: int, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.mha_face_to_voice = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.mha_voice_to_face = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.gate_face = nn.Parameter(torch.zeros(1))
        self.gate_voice = nn.Parameter(torch.zeros(1))
        self.norm_face = nn.LayerNorm(embed_dim)
        self.norm_voice = nn.LayerNorm(embed_dim)

    def forward(self, face_feat: torch.Tensor, voice_feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # face_feat: (B, 1, D), voice_feat: (B, 1, D)
        # Face queries Voice
        face_ctx, _ = self.mha_face_to_voice(query=face_feat, key=voice_feat, value=voice_feat)
        face_fused = self.norm_face(face_feat + torch.tanh(self.gate_face) * face_ctx)

        # Voice queries Face
        voice_ctx, _ = self.mha_voice_to_face(query=voice_feat, key=face_feat, value=face_feat)
        voice_fused = self.norm_voice(voice_feat + torch.tanh(self.gate_voice) * voice_ctx)

        return face_fused.squeeze(1), voice_fused.squeeze(1)


class SubCenterArcFace(nn.Module):
    """
    Additive Angular Margin Softmax (AAM-Softmax / ArcFace) on Hypersphere S^(D-1).
    Margin m = 0.35, Scale s = 32.0.
    """
    def __init__(self, in_features: int, num_classes: int, scale: float = 32.0, margin: float = 0.35):
        super().__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, x: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
        # x is normalized L2 feature
        cosine = F.linear(F.normalize(x, p=2, dim=1), F.normalize(self.weight, p=2, dim=1))
        sine = torch.sqrt(torch.clamp(1.0 - torch.pow(cosine, 2), min=1e-7))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros(cosine.size(), device=x.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1.0)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        return output


### 🏛️ 7. Full Billion-Scale Foundation Multimodal Architecture


In [ ]:
from transformers import AutoModel, AutoFeatureExtractor

class FLAGBillionOmniModel(nn.Module):
    """
    Billion-Parameter Multilingual Multimodal Foundation System for FLAG 2027.
    Backbones:
      - Speech: Meta MMS-1B-All (1,400+ languages, 1B params)
      - Vision: Meta DINOv2-Large / Giant (Self-supervised ViT)
    """
    def __init__(
        self,
        num_speakers: int = 100,
        audio_backbone: str = cfg.AUDIO_BACKBONE,
        vision_backbone: str = cfg.VISION_BACKBONE,
        embed_dim: int = cfg.SHARED_EMBED_DIM
    ):
        super().__init__()
        self.embed_dim = embed_dim
        print(f"[Model Init] Loading Audio Foundation Backbone: {audio_backbone}...")
        self.audio_encoder = AutoModel.from_pretrained(audio_backbone)
        
        # Audio Hidden Dim Discovery
        audio_hidden = getattr(self.audio_encoder.config, "hidden_size", cfg.AUDIO_FEAT_DIM)
        num_audio_layers = getattr(self.audio_encoder.config, "num_hidden_layers", 24)

        if cfg.GRADIENT_CHECKPOINTING and hasattr(self.audio_encoder, "gradient_checkpointing_enable"):
            self.audio_encoder.gradient_checkpointing_enable()

        # Freeze raw waveform feature extraction CNN for stability
        if hasattr(self.audio_encoder, "feature_extractor"):
            for p in self.audio_encoder.feature_extractor.parameters():
                p.requires_grad = False

        self.audio_aggregator = MultiLayerSoftmaxPooling(num_layers=num_audio_layers, hidden_dim=audio_hidden)
        self.audio_asp = AttentiveStatisticsPooling(input_dim=audio_hidden, hidden_dim=256)
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_hidden * 2, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.PReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

        print(f"[Model Init] Loading Vision Foundation Backbone: {vision_backbone}...")
        self.vision_encoder = AutoModel.from_pretrained(vision_backbone)
        vision_hidden = getattr(self.vision_encoder.config, "hidden_size", cfg.VISION_FEAT_DIM)

        if cfg.GRADIENT_CHECKPOINTING and hasattr(self.vision_encoder, "gradient_checkpointing_enable"):
            self.vision_encoder.gradient_checkpointing_enable()

        self.vision_proj = nn.Sequential(
            nn.Linear(vision_hidden, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.PReLU(),
            nn.Linear(embed_dim, embed_dim)
        )

        # Gated Cross-Modal Attention
        self.cross_attn = GatedCrossModalAttention(embed_dim=embed_dim, num_heads=cfg.CROSS_ATTN_HEADS)

        # Adversarial Demographic Debiasing
        self.grl = GradientReversalLayer(alpha=1.0)
        self.gender_classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 2)
        )

        # ArcFace Metric Learning Heads
        self.arcface_face = SubCenterArcFace(embed_dim, num_speakers, scale=cfg.ARCFACE_SCALE, margin=cfg.ARCFACE_MARGIN)
        self.arcface_voice = SubCenterArcFace(embed_dim, num_speakers, scale=cfg.ARCFACE_SCALE, margin=cfg.ARCFACE_MARGIN)

    def extract_face(self, face_img: torch.Tensor) -> torch.Tensor:
        v_out = self.vision_encoder(pixel_values=face_img)
        # Use CLS token for face morphology
        cls_token = v_out.last_hidden_state[:, 0, :]
        feat = self.vision_proj(cls_token)
        return feat

    def extract_voice(self, voice_wav: torch.Tensor) -> torch.Tensor:
        a_out = self.audio_encoder(voice_wav, output_hidden_states=True)
        if a_out.hidden_states is not None:
            seq_feat = self.audio_aggregator(a_out.hidden_states)
        else:
            seq_feat = a_out.last_hidden_state
        pooled = self.audio_asp(seq_feat)
        feat = self.audio_proj(pooled)
        return feat

    def forward(
        self,
        face_img: torch.Tensor,
        voice_wav: torch.Tensor,
        spk_label: Optional[torch.Tensor] = None,
        grl_lambda: float = 1.0
    ) -> Dict[str, torch.Tensor]:
        f_feat = self.extract_face(face_img)
        v_feat = self.extract_voice(voice_wav)

        # Cross-Modal Fusion
        f_fused, v_fused = self.cross_attn(f_feat.unsqueeze(1), v_feat.unsqueeze(1))

        # Project to Hypersphere S^(D-1)
        e_f = F.normalize(f_fused, p=2, dim=-1)
        e_v = F.normalize(v_fused, p=2, dim=-1)

        # Exact CodaBench Euclidean Distance: sqrt(2 - 2 * cos(θ))
        # Equivalent to ||e_f - e_v||_2 on unit hypersphere
        cos_sim = torch.sum(e_f * e_v, dim=-1).clamp(-1.0, 1.0)
        dist = torch.sqrt(torch.clamp(2.0 - 2.0 * cos_sim, min=1e-8))

        out = {
            "face_embed": e_f,
            "voice_embed": e_v,
            "distance": dist,
            "cosine_sim": cos_sim
        }

        # Training heads
        if spk_label is not None:
            out["arc_face_logits"] = self.arcface_face(e_f, spk_label)
            out["arc_voice_logits"] = self.arcface_voice(e_v, spk_label)

            # Adversarial Gender Debiasing
            f_rev = self.grl(e_f, grl_lambda)
            v_rev = self.grl(e_v, grl_lambda)
            out["face_gender_logits"] = self.gender_classifier(f_rev)
            out["voice_gender_logits"] = self.gender_classifier(v_rev)

        return out

print("Billion-Scale Foundation Architecture compiled successfully.")


### 🎯 8. Multi-Task Objective Function & Cosine Optimization


In [ ]:
class FLAGBillionLoss(nn.Module):
    """
    Composite Loss Function:
      L = L_ArcFace(Face) + L_ArcFace(Voice)
        + λ_cont * L_CrossModal_Contrastive
        + λ_grl  * L_Gender_Adversarial
        + λ_orth * L_Subspace_Orthogonality
    """
    def __init__(self, cfg: BillionConfig):
        super().__init__()
        self.cfg = cfg
        self.ce = nn.CrossEntropyLoss()

    def contrastive_loss(self, e_f: torch.Tensor, e_v: torch.Tensor) -> torch.Tensor:
        # Symmetric InfoNCE / NT-Xent loss
        sim_matrix = torch.matmul(e_f, e_v.T) / self.cfg.CONTRASTIVE_TEMP
        labels = torch.arange(e_f.size(0), device=e_f.device)
        loss_f2v = self.ce(sim_matrix, labels)
        loss_v2f = self.ce(sim_matrix.T, labels)
        return (loss_f2v + loss_v2f) / 2.0

    def orthogonal_loss(self, e_id: torch.Tensor, e_gen: torch.Tensor) -> torch.Tensor:
        # Drives inner product <z_id, z_gen> to zero
        sim = torch.sum(e_id * e_gen, dim=-1)
        return torch.mean(sim ** 2)

    def forward(
        self,
        outputs: Dict[str, torch.Tensor],
        spk_label: torch.Tensor,
        gender_label: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        loss_arc_f = self.ce(outputs["arc_face_logits"], spk_label)
        loss_arc_v = self.ce(outputs["arc_voice_logits"], spk_label)
        loss_id = (loss_arc_f + loss_arc_v) / 2.0

        loss_cont = self.contrastive_loss(outputs["face_embed"], outputs["voice_embed"])

        loss_grl_f = self.ce(outputs["face_gender_logits"], gender_label)
        loss_grl_v = self.ce(outputs["voice_gender_logits"], gender_label)
        loss_grl = (loss_grl_f + loss_grl_v) / 2.0

        total_loss = (
            self.cfg.LAMBDA_ARCFACE * loss_id +
            self.cfg.LAMBDA_CONTRASTIVE * loss_cont +
            self.cfg.LAMBDA_GRL * loss_grl
        )

        return {
            "total_loss": total_loss,
            "loss_id": loss_id,
            "loss_cont": loss_cont,
            "loss_grl": loss_grl,
        }


### 🚀 9. Training Engine with Dynamic GRL Lambda Scheduling & AMP


In [ ]:
class EarlyStopping:
    def __init__(
        self,
        patience: int = cfg.EARLY_STOPPING_PATIENCE,
        min_delta: float = cfg.EARLY_STOPPING_MIN_DELTA,
        mode: str = cfg.EARLY_STOPPING_MODE,
        restore_best_weights: bool = cfg.EARLY_STOPPING_RESTORE_BEST,
        verbose: bool = True
    ):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode.lower()
        self.restore_best_weights = restore_best_weights
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
        self.best_state_dict = None

    def is_better(self, score: float) -> bool:
        if self.best_score is None:
            return True
        if self.mode == "min":
            return score < (self.best_score - self.min_delta)
        else:
            return score > (self.best_score + self.min_delta)

    def step(self, score: float, model: nn.Module, epoch: int) -> bool:
        if math.isnan(score) or math.isinf(score):
            return False
        if self.is_better(score):
            if self.verbose:
                if self.best_score is not None:
                    print(f"[EarlyStopping] Metric improved from {self.best_score:.4f} to {score:.4f} at epoch {epoch}. Resetting patience.")
                else:
                    print(f"[EarlyStopping] Baseline metric recorded: {score:.4f} at epoch {epoch}.")
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
            if self.restore_best_weights:
                self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return True
        else:
            self.counter += 1
            if self.verbose:
                print(f"[EarlyStopping] Patience: {self.counter}/{self.patience} (Best: {self.best_score:.4f} at epoch {self.best_epoch})")
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print(f"[EarlyStopping] 🛑 Early stopping triggered! Validation performance stagnated for {self.patience} epochs.")
            return False

    def restore(self, model: nn.Module):
        if self.restore_best_weights and self.best_state_dict is not None:
            model.load_state_dict(self.best_state_dict)
            if self.verbose:
                print(f"[EarlyStopping] Restored best model weights from epoch {self.best_epoch} (Score: {self.best_score:.4f}).")


def train_flag_billion(model, train_dataset, val_dataset=None, cfg=cfg):
    if len(train_dataset) == 0:
        print("[Warning] Train dataset is empty. Skipping training loop.")
        return model

    dataloader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=True
    )

    val_dataloader = None
    if val_dataset is not None and len(val_dataset) > 0:
        val_dataloader = DataLoader(
            val_dataset,
            batch_size=cfg.BATCH_SIZE,
            shuffle=False,
            num_workers=cfg.NUM_WORKERS,
            pin_memory=torch.cuda.is_available()
        )

    criterion = FLAGBillionLoss(cfg)
    early_stopping = EarlyStopping(
        patience=cfg.EARLY_STOPPING_PATIENCE,
        min_delta=cfg.EARLY_STOPPING_MIN_DELTA,
        mode=cfg.EARLY_STOPPING_MODE,
        restore_best_weights=cfg.EARLY_STOPPING_RESTORE_BEST
    )

    backbone_params = list(model.audio_encoder.parameters()) + list(model.vision_encoder.parameters())
    head_params = [p for n, p in model.named_parameters() if not n.startswith("audio_encoder") and not n.startswith("vision_encoder")]

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg.LR_BACKBONE, "weight_decay": cfg.WEIGHT_DECAY},
        {"params": head_params, "lr": cfg.LR_HEADS, "weight_decay": cfg.WEIGHT_DECAY}
    ])

    total_steps = len(dataloader) * cfg.NUM_EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda') if (cfg.USE_AMP and torch.cuda.is_available()) else None

    print(f"[Training] Starting optimization for {cfg.NUM_EPOCHS} epochs (Early Stopping Patience: {cfg.EARLY_STOPPING_PATIENCE})...")
    best_loss = float("inf")

    for epoch in range(1, cfg.NUM_EPOCHS + 1):
        model.train()
        epoch_losses = []
        pbar = tqdm(dataloader, desc=f"Epoch {epoch:02d}/{cfg.NUM_EPOCHS:02d}", leave=False)

        for step, batch in enumerate(pbar):
            p = float(step + (epoch - 1) * len(dataloader)) / total_steps
            grl_lambda = 2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0

            face_img = batch["face_img"].to(cfg.DEVICE, non_blocking=True)
            voice_wav = batch["voice_wav"].to(cfg.DEVICE, non_blocking=True)
            spk_label = batch["spk_label"].to(cfg.DEVICE, non_blocking=True)
            gender_label = batch["gender_label"].to(cfg.DEVICE, non_blocking=True)

            optimizer.zero_grad()

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    outputs = model(face_img, voice_wav, spk_label=spk_label, grl_lambda=grl_lambda)
                    losses = criterion(outputs, spk_label, gender_label)
                    loss = losses["total_loss"] / cfg.GRAD_ACCUM_STEPS

                scaler.scale(loss).backward()
                if (step + 1) % cfg.GRAD_ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
            else:
                outputs = model(face_img, voice_wav, spk_label=spk_label, grl_lambda=grl_lambda)
                losses = criterion(outputs, spk_label, gender_label)
                loss = losses["total_loss"] / cfg.GRAD_ACCUM_STEPS
                loss.backward()
                if (step + 1) % cfg.GRAD_ACCUM_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP_NORM)
                    optimizer.step()
                    scheduler.step()

            epoch_losses.append(losses["total_loss"].item())
            pbar.set_postfix({
                "loss": f"{losses['total_loss'].item():.3f}",
                "id": f"{losses['loss_id'].item():.3f}",
                "grl": f"{losses['loss_grl'].item():.3f}"
            })

        mean_train_loss = np.mean(epoch_losses) if epoch_losses else 0.0

        # Validation Step for Early Stopping
        val_losses = []
        if val_dataloader is not None:
            model.eval()
            with torch.no_grad():
                for v_batch in val_dataloader:
                    v_face = v_batch["face_img"].to(cfg.DEVICE, non_blocking=True)
                    v_voice = v_batch["voice_wav"].to(cfg.DEVICE, non_blocking=True)
                    v_spk = v_batch["spk_label"].to(cfg.DEVICE, non_blocking=True)
                    v_gen = v_batch["gender_label"].to(cfg.DEVICE, non_blocking=True)
                    v_out = model(v_face, v_voice, spk_label=v_spk)
                    v_loss = criterion(v_out, v_spk, v_gen)
                    val_losses.append(v_loss["total_loss"].item())
            mean_val_loss = np.mean(val_losses) if val_losses else mean_train_loss
        else:
            mean_val_loss = mean_train_loss

        val_str = f"Val Loss: {mean_val_loss:.4f}" if val_dataloader is not None else "Val: N/A"
        print(f"Epoch {epoch:02d} | Train Loss: {mean_train_loss:.4f} | {val_str} | GRL Lambda: {grl_lambda:.3f}")

        # Early Stopping Check & Checkpoint
        is_best = early_stopping.step(mean_val_loss, model, epoch)
        if is_best:
            best_loss = mean_val_loss
            torch.save(model.state_dict(), cfg.BEST_MODEL_PATH)
            print(f" -> Saved new best model checkpoint to {cfg.BEST_MODEL_PATH}")

        if early_stopping.early_stop:
            print(f"\n[EarlyStopping] Terminating training early to prevent overfitting!")
            early_stopping.restore(model)
            break

    return model


### 📊 10. Complete 4-Cell Benchmark Evaluation (EER & AUC)


In [ ]:
def compute_eer(dists: np.ndarray, labels: np.ndarray) -> Tuple[float, float]:
    valid = (labels == 0) | (labels == 1)
    if not np.any(valid):
        return -1.0, -1.0
    v_dists = dists[valid]
    v_labels = labels[valid]
    
    # Invert distance for ROC where high score indicates match
    sim = -v_dists
    fpr, tpr, _ = roc_curve(v_labels, sim, pos_label=1)
    fnr = 1.0 - tpr
    try:
        eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    except Exception:
        idx = np.nanargmin(np.absolute(fnr - fpr))
        eer = (fpr[idx] + fnr[idx]) / 2.0
    roc_auc = auc(fpr, tpr)
    return float(eer * 100.0), float(roc_auc * 100.0)


def evaluate_trial_cell(model, trial_path, dev_dir, output_txt, cfg):
    if not os.path.exists(trial_path):
        print(f"[Warning] Trial file {trial_path} not found.")
        return -1.0, -1.0

    dataset = FLAGBillionDevDataset(trial_path, dev_dir)
    if len(dataset) == 0:
        print(f"[Warning] 0 trials loaded from {trial_path}.")
        return -1.0, -1.0

    loader = DataLoader(dataset, batch_size=cfg.BATCH_SIZE * 2, shuffle=False, num_workers=cfg.NUM_WORKERS)
    model.eval()

    all_pairs = []
    all_dists = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            face_img = batch["face_img"].to(cfg.DEVICE, non_blocking=True)
            voice_wav = batch["voice_wav"].to(cfg.DEVICE, non_blocking=True)
            b_dists = model(face_img, voice_wav)["distance"].cpu().numpy()
            
            all_pairs.extend(batch["pair_id"])
            all_dists.extend(b_dists.tolist())
            all_labels.extend(batch["label"].numpy().tolist())

    dists_np = np.array(all_dists)
    labels_np = np.array(all_labels)

    # Write CodaBench submission text: [pair_id] [distance]
    os.makedirs(os.path.dirname(output_txt), exist_ok=True)
    with open(output_txt, "w", encoding="utf-8") as f:
        for p, d in zip(all_pairs, dists_np):
            f.write(f"{p} {d:.6f}\n")

    eer, roc_auc = compute_eer(dists_np, labels_np)
    return eer, roc_auc


def run_full_codabench_eval(model, dev_dir, cfg):
    print("=" * 80)
    print("FLAG 2027 Challenge: 4-Cell CodaBench Evaluation")
    print("=" * 80)

    cells = [
        ("gender / English_heard",
         os.path.join(dev_dir, "gender", "trials_v4_English_heard.txt"),
         cfg.SUBMISSION_GENDER_ENGLISH),
        ("gender / Bangla_unheard",
         os.path.join(dev_dir, "gender", "trials_v4_Bangla_unheard.txt"),
         cfg.SUBMISSION_GENDER_BANGLA),
        ("no_gender / English_heard",
         os.path.join(dev_dir, "no_gender", "trials_v4_English_heard.txt"),
         cfg.SUBMISSION_NO_GENDER_ENGLISH),
        ("no_gender / Bangla_unheard",
         os.path.join(dev_dir, "no_gender", "trials_v4_Bangla_unheard.txt"),
         cfg.SUBMISSION_NO_GENDER_BANGLA),
    ]

    results = []
    for cell_name, trial_p, out_p in cells:
        # Fallback to search if specific name varies
        if not os.path.exists(trial_p):
            cell_folder = os.path.dirname(trial_p)
            candidates = glob.glob(os.path.join(cell_folder, "*.txt"))
            if candidates:
                trial_p = candidates[0]

        eer, roc_auc = evaluate_trial_cell(model, trial_p, dev_dir, out_p, cfg)
        results.append((cell_name, eer, roc_auc))

    print(f"\n{'Evaluation Cell':<32} | {'EER (%)':<10} | {'AUC (%)':<10}")
    print("-" * 60)
    valid_eers = []
    for name, eer, roc_auc in results:
        eer_str = f"{eer:.2f}%" if eer >= 0 else "N/A (Blind)"
        auc_str = f"{roc_auc:.2f}%" if roc_auc >= 0 else "N/A"
        print(f"{name:<32} | {eer_str:<10} | {auc_str:<10}")
        if eer >= 0:
            valid_eers.append(eer)

    if valid_eers:
        mean_eer = np.mean(valid_eers)
        print("-" * 60)
        print(f"{'OVERALL AVERAGE EER':<32} | {mean_eer:.2f}%")
    print("=" * 80)


### 📦 11. CodaBench Submission Packaging (`submission.zip`)


In [ ]:
def package_codabench_submission(cfg: BillionConfig):
    """
    Strictly verifies and packages the 4 output text files into submission.zip
    """
    required_files = [
        (cfg.SUBMISSION_GENDER_ENGLISH, "gender/sub_score_v4_English_heard.txt"),
        (cfg.SUBMISSION_GENDER_BANGLA, "gender/sub_score_v4_Bangla_unheard.txt"),
        (cfg.SUBMISSION_NO_GENDER_ENGLISH, "no_gender/sub_score_v4_English_heard.txt"),
        (cfg.SUBMISSION_NO_GENDER_BANGLA, "no_gender/sub_score_v4_Bangla_unheard.txt"),
    ]

    print("[Submission] Validating submission text files...")
    all_ok = True
    for abs_p, rel_p in required_files:
        if os.path.exists(abs_p) and os.path.getsize(abs_p) > 0:
            with open(abs_p, "r", encoding="utf-8") as f:
                lines = f.readlines()
            print(f" -> Found {rel_p}: {len(lines)} scored pairs.")
        else:
            print(f" -> [Missing/Empty] {rel_p}")
            all_ok = False

    if not all_ok:
        print("[Warning] Some submission files are missing. Creating dummy entries if needed for packaging test.")

    print(f"[Submission] Generating zip: {cfg.SUBMISSION_ZIP_PATH}...")
    with zipfile.ZipFile(cfg.SUBMISSION_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
        for abs_p, rel_p in required_files:
            if os.path.exists(abs_p):
                zipf.write(abs_p, arcname=rel_p)

    zip_size_kb = os.path.getsize(cfg.SUBMISSION_ZIP_PATH) / 1024.0
    print(f"✅ Success! Generated {cfg.SUBMISSION_ZIP_PATH} ({zip_size_kb:.2f} KB).")
    print("This zip file is ready for direct upload to CodaBench FLAG 2027 Challenge!")


### 🏁 12. Main Execution Pipeline


In [ ]:
# 1. Initialize Dataset & Train/Val Split for Early Stopping
train_dataset, val_dataset = create_billion_train_val_datasets(TRAIN_DIR, cfg.VAL_SPLIT_RATIO, cfg.SEED)
num_spks = max(100, len(train_dataset.speaker_to_id))

# 2. Build Billion-Scale Multimodal Foundation Model
model = FLAGBillionOmniModel(
    num_speakers=num_spks,
    audio_backbone=cfg.AUDIO_BACKBONE,
    vision_backbone=cfg.VISION_BACKBONE,
    embed_dim=cfg.SHARED_EMBED_DIM
).to(cfg.DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 70)
print(f"Total Parameters     : {total_params / 1e9:.2f} Billion ({total_params:,})")
print(f"Trainable Parameters : {trainable_params / 1e6:.2f} Million ({trainable_params:,})")
print("=" * 70)

# 3. Train Model with Early Stopping
if len(train_dataset) > 0:
    model = train_flag_billion(model, train_dataset, val_dataset, cfg)

# 4. Evaluate Across All 4 CodaBench Cells
if DEV_DIR and os.path.exists(DEV_DIR):
    run_full_codabench_eval(model, DEV_DIR, cfg)

# 5. Package Submission
package_codabench_submission(cfg)
